In [1]:
import sys
sys.path.append('/home/lytq/Spatial-Transcriptomics-Benchmark/utils')
from sdmbench import compute_ARI, compute_NMI, compute_CHAOS, compute_PAS, compute_ASW, compute_HOM, compute_COM

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is dep

In [4]:
pred_keys = {
    'BayesSpace': 'spatial.cluster',
    'conST': 'conST_leiden',
    'SEDR': 'SEDR',
    'Seurat': 'seurat_clusters',
    'SpaceFlow': 'pred',
    'SpaGCN': 'refined_pred',
    'STAGATE': 'STAGATE',
    'stLearn': 'X_pca_kmeans',
}

umap_keys = {
    'BayesSpace': ['UMAP1', 'UMAP2'],
    'conST': ['UMAP1', 'UMAP2'],
    'SEDR': ['UMAP1', 'UMAP2'],
    'Seurat': ['umap_1', 'umap_2'],
    'SpaceFlow': ['UMAP1', 'UMAP2'],
    'SpaGCN': ['UMAP1', 'UMAP2'],
    'STAGATE': ['UMAP1', 'UMAP2'],
    'stLearn': ['UMAP1', 'UMAP2'],
}

methods = ['BayesSpace', 'conST', 'SEDR', 'Seurat', 'SpaceFlow', 'SpaGCN', 'STAGATE', 'stLearn']

SEEDS = [42, 123, 456, 789, 2024]

dataset = 'DLPFC'

data_folder = f'../../data/{dataset}'
input_dir = f'../../Results/'
output_dir = f'../../results/paper2/'
os.makedirs(output_dir, exist_ok=True)

In [5]:
for method in methods:
    if method not in ['Seurat', 'stLearn']:
        continue
    pred_key = pred_keys[method]
    umap_key = umap_keys[method]

    print(f'================= Processing {method} {dataset} =================')
    for seed in SEEDS:
        print(f'================== Seed {seed} =================')
        input_path = os.path.join(input_dir, f'{seed}/{dataset}/{method}')
        files = glob.glob(input_path + '/*')
        files = [f for f in files if os.path.isdir(f)]

        metrics_list = []
        for i, file in enumerate(files):
            section_id = file.split('/')[-1]
            out_path = os.path.join(output_dir, f'{seed}/{dataset}/{method}/{section_id}')
            os.makedirs(out_path, exist_ok=True)
            
            print(f'[{i + 1}] Processing {section_id}')

            file_path = os.path.join(data_folder, section_id)

            # Load dataset
            adata = sc.read_visium(file_path, count_file='filtered_feature_bc_matrix.h5', load_images=True)
            adata.var_names_make_unique()
            metadata = pd.read_csv(file + '/cell_metadata.csv', index_col=0)
            metadata.to_csv(os.path.join(out_path, 'cell_metadata.csv'))
            gt_metadata = pd.read_csv(os.path.join(file_path, 'metadata.tsv'), sep='\t')

            if method not in ['SpaceFlow', 'SpaGCN']:
                metadata = metadata.loc[adata.obs.index]
            gt_metadata = gt_metadata.loc[adata.obs.index]

            # Match adata and metadata 
            # adata = adata[adata.obs.index.isin(metadata.index)]
            
            umap_coords = pd.read_csv(file + '/spatial_umap_coords.csv')
            umap_coords.to_csv(os.path.join(out_path, 'spatial_umap_coords.csv'))

            low_dim = pd.read_csv(file + '/low_dim_data.csv')
            low_dim.to_csv(os.path.join(out_path, 'low_dim_data.csv'), index=False)

            metrics = pd.read_csv(file + '/metrics.csv', index_col=0)
            
            adata.obs['gt'] = gt_metadata['layer_guess'].values
            pred = metadata[pred_key].values
            if min(pred) == 0:
                pred += 1

            adata.obs['pred'] = pred.astype(str)
            adata.obsm['X_umap'] = umap_coords[umap_key].values
            adata = adata[~pd.isnull(adata.obs['gt'])]
            adata.obs['gt'] = adata.obs['gt'].astype(str)

            time_taken, memmory_used = metrics['Time'].values[0], metrics['Memory'].values[0]
            results = {
                'SEED': seed,
                method: section_id,
                'ARI': compute_ARI(adata, 'gt', 'pred'),
                'AMI': compute_NMI(adata, 'gt', 'pred'),
                'Homogeneity': compute_HOM(adata, 'gt', 'pred'),
                'Completeness': compute_COM(adata, 'gt', 'pred'),
                'ASW': compute_ASW(adata, 'pred'),
                'CHAOS': compute_CHAOS(adata, 'pred'),
                'PAS': compute_PAS(adata, 'pred'),
                'Time (s)': time_taken,
                'Memory (MB)': memmory_used
            }
            df_results = pd.DataFrame([results])
            df_results.to_csv(os.path.join(out_path, 'metrics.csv'), index=False)
            metrics_list.append(results)

            # print(f'    Results saved to {out_path}')

        df_metrics = pd.DataFrame(metrics_list)
        df_metrics = df_metrics.sort_values(by=method)
        df_metrics.to_csv(os.path.join(output_dir, f'{seed}/{dataset}/{method}', 'metrics.csv'), index=False)
        print(f'================= Finished {seed} {method} {dataset} =================')

================= Processing Seurat DLPFC =================
================== Seed 42 =================
[1] Processing 151669
[2] Processing 151671
[3] Processing 151675
[4] Processing 151672
[5] Processing 151674
[6] Processing 151673
[7] Processing 151670
[8] Processing 151510
[9] Processing 151509
[10] Processing 151507
[11] Processing 151508
[12] Processing 151676
================= Finished 42 Seurat DLPFC =================
================== Seed 123 =================
[1] Processing 151669
[2] Processing 151671
[3] Processing 151675
[4] Processing 151672
[5] Processing 151674
[6] Processing 151673
[7] Processing 151670
[8] Processing 151510
[9] Processing 151509
[10] Processing 151507
[11] Processing 151508
[12] Processing 151676
================= Finished 123 Seurat DLPFC =================
================== Seed 456 =================
[1] Processing 151669
[2] Processing 151671
[3] Processing 151675
[4] Processing 151672
[5] Processing 151674
[6] Processing 151673
[7] Processing

## Combine all metrics

In [6]:
all_metrics = []
output_path = f'../../results/paper2/dlpfc_metrics.csv'

for seed in SEEDS:
    print(seed)
    input_dir = f'../../results/paper2/{seed}/{dataset}'
    input_files = glob.glob(input_dir + '/*')
    input_files = [f for f in input_files if os.path.isdir(f)]

    for file in input_files:
        df_metrics = pd.read_csv(os.path.join(file, 'metrics.csv'))
        all_metrics.append(df_metrics)

# all_metrics = sorted(
#     all_metrics,
#     key=lambda x: (x['Method'].iloc[0], x['SEED'].iloc[0])
# )

with open(output_path, 'w') as f:
    print(f'Writing results to {output_path}')
    for df in all_metrics:
        df.to_csv(f, index=False, header=True) 
        # df.to_csv(f, index=False, header=not f.tell())

42
123
456
789
2024
Writing results to ../../results/paper2/dlpfc_metrics.csv


In [92]:
for df in all_metrics:
    print(df)

    SEED   conST       ARI       AMI  Homogeneity  Completeness       ASW  \
0     42  151507  0.367075  0.522545     0.531669      0.513729 -0.048012   
1     42  151508  0.206588  0.346179     0.360258      0.333159 -0.098081   
2     42  151509  0.308722  0.412969     0.444891      0.385322 -0.070320   
3     42  151510  0.282527  0.393777     0.427969      0.364644 -0.081671   
4     42  151669  0.500988  0.608608     0.654996      0.568357  0.033891   
5     42  151670  0.348891  0.537066     0.618249      0.474730  0.100666   
6     42  151671  0.423462  0.574307     0.500481      0.673681  0.196374   
7     42  151672  0.569288  0.621416     0.642277      0.601868  0.084279   
8     42  151673  0.514972  0.683812     0.694009      0.673910  0.012738   
9     42  151674  0.348359  0.512404     0.517924      0.507001 -0.018945   
10    42  151675  0.386447  0.529574     0.532483      0.526697 -0.008622   
11    42  151676  0.476940  0.645086     0.657988      0.632681  0.034648   